# SAHYOG SAR - full run on Kaggle (free GPU)
SIH 2026 - PS 26177. Trains the deployment detector on **real** data (AIResQ + HIT-UAV + SARD)
and runs the whole repo. GPU only needed once, ~1-3 h; the trained model is saved and reused.

## Before running: two settings (right panel)
1. **Accelerator -> GPU T4 x2** (the quota line must show 'X hrs / 30 hrs')
2. **Internet -> On**  (off by default)
Recommended: **Persistence -> Output files** so trained weights survive 'Save & Run All'.


## 0. Download AIResQ (7.0 GB zip, open access - Zenodo record 19677785)
Open access (CC-BY-4.0), no request needed. md5 = 9207302d8c73b26ee22a7c40f0f46c39.
Disk note: /kaggle/working caps at 20 GB. Delete the zip after unzipping.


In [ ]:
import os
os.chdir('/kaggle/working')
!wget -c -O AIResQ.zip 'https://zenodo.org/records/19677785/files/AIResQ.zip?download=1'
!md5sum AIResQ.zip        # expect 9207302d8c73b26ee22a7c40f0f46c39
!unzip -q AIResQ.zip -d datasets-airesq && rm AIResQ.zip
!ls datasets-airesq


## 1. Clone the repo + install


In [ ]:
os.chdir('/kaggle/working')
!git clone https://github.com/ShashankRoy-iit/SIH.git
os.chdir('/kaggle/working/SIH')
!pip install -q -e '.[sim,dev]' onnx onnxruntime
!pip install -q ultralytics
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))


## 2. Sanity check - the whole project runs


In [ ]:
!python scripts/doctor.py
!python -m pytest tests/ -q
!python scripts/run_rescue_simulation.py --scenario flood --duration 460


## 3. HIT-UAV (aerial LWIR) - attach via Add Input, then convert labels
Add the HIT-UAV **dataset** (not a notebook) via Data -> Add Input. It lands read-only
under /kaggle/input/. HIT-UAV ships YOLO-format txt (class xc yc w h), ids:
0 Person, 1 Car, 2 Bicycle, 3 OtherVehicle, 4 DontCare. Map to the repo's 6 classes.


In [ ]:
import shutil, glob
from pathlib import Path

SRC = '/kaggle/input/hit-uav'          # adjust to your attached dataset name
DST = Path('datasets/hit-uav')
MAP = {0: 0, 1: 2, 2: 2, 3: 2}        # person->person(0); car/bicycle/other->vehicle(2); drop 4
for split in ('train', 'test', 'val'):
    for d in (DST/'images'/split, DST/'labels'/split):
        d.mkdir(parents=True, exist_ok=True)
    for im in glob.glob(f'{SRC}/images/{split}/*.jpg'):
        shutil.copy(im, DST/'images'/split)
        lab = Path(im.replace('/images/', '/labels/')).with_suffix('.txt')
        out = []
        if lab.is_file():
            for line in lab.read_text().splitlines():
                p = line.split()
                if p and int(p[0]) in MAP:
                    out.append(f"{MAP[int(p[0])]} " + ' '.join(p[1:]))
        (DST/'labels'/split/(Path(im).stem + '.txt')).write_text('\n'.join(out))
print('HIT-UAV images:', len(glob.glob(str(DST/'images/train/*.jpg'))))


## 4. AIResQ - inspect layout, then convert to YOLO (format varies by release)
AIResQ labels may be COCO JSON, CSV, or txt. Inspect first, then convert. If COCO:
use ultralytics' built-in converter, or JSON2YOLO. This cell prints the layout.


In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/working/datasets-airesq'):
    if files:
        depth = root.replace('/kaggle/working/datasets-airesq', '').count('/')
        if depth <= 2:
            print(root, '->', files[:6], f'({len(files)} files)')


If AIResQ labels are COCO JSON, convert like this (adjust paths):


In [ ]:
# from ultralytics.data.converter import convert_coco
# convert_coco(labels_dir='datasets-airesq/annotations/',
#              use_segments=False, use_keypoints=False, cls91to80=False)
# then move images+labels into datasets/airesq/{images,labels}/{train,val}


## 5. Train on the GPU (the step Codespaces cannot do)


In [ ]:
# Thermal (LWIR) - the deployment detector. --p2 resolves configs/models/yolo11n-p2.yaml
!python scripts/train_detector.py --train --data datasets/hit-uav/data.yaml \
    --model yolo11n.yaml --p2 --channels 1 --imgsz 640 --epochs 120 --batch 16 \
    --project /kaggle/working/runs --name thermal --device 0

# RGB (phone camera) - SARD postures, or the repo's sim-rgb as a start
!python scripts/train_detector.py --train --data datasets/sim-rgb/data.yaml \
    --model yolo11n.yaml --imgsz 640 --epochs 120 --batch 16 \
    --project /kaggle/working/runs --name rgb --device 0


## 6. Export the three deployable files


In [ ]:
# flight model (ONNX, opset 12) - this name is already in the registry
!python scripts/export_model.py --weights /kaggle/working/runs/thermal/weights/best.pt \
    --out models/yolo11n-thermal-sar.onnx --imgsz 640 512 --channels 3
# phone model (TFLite INT8, Hexagon DSP)
!python scripts/export_phone_model.py --weights /kaggle/working/runs/rgb/weights/best.pt \
    --out models/person-nano-int8.tflite
# QNN .bin for RB3 Gen 2 (prints the command; needs a Qualcomm AI Hub token)
!python scripts/export_model.py --qnn-recipe


## 7. Register, verify, evaluate


In [ ]:
!python scripts/fetch_models.py --verify-all
!SAR_DETECTOR=auto python scripts/eval_detector.py --mode both --json artifacts/detector_eval_neural.json


## 8. Keep the results (Kaggle wipes /kaggle/working at session end)
- Right panel -> **Output** -> tick + **Download**: best.pt, models/*.onnx, *.tflite, artifacts/*.json
- Or pin them as a Dataset version for the next session.
- **GitHub**: commit ONLY artifacts/train_results.json + artifacts/detector_eval_neural.json
  to the branch arena/01a0965a-sih (never push model files - they are gitignored).
